获取stock资金流向数据

获取一只或者多只股票在一个时间段内的资金流向数据

**调用方法**

```python
from jqdata import *
get_money_flow(security_list, start_date=None, end_date=None, fields=None, count=None)
```

**参数**

- security_list: 一只股票代码或者一个股票代码的 list
- start_date: 开始日期, 一个字符串或者 [datetime.datetime]/[datetime.date] 对象
- end_date: 结束日期, 一个字符串或者 [datetime.date]/[datetime.datetime] 对象
- count: 数量, 与 start_date 二选一，不可同时使用, 必须大于 0. 表示返回 end_date 之前 count 个交易日的数据, 包含 end_date
- fields: 字段名或者 list, 可选. 默认为 None, 表示取全部字段, 各字段含义如下：

| 字段名          | 含义            | 备注                                                        |
| :-------------- | :-------------- | :---------------------------------------------------------- |
| date            | 日期            |                                                             |
| sec_code        | 股票代码        |                                                             |
| change_pct      | 涨跌幅(%)       |                                                             |
| net_amount_main | 主力净额(万)    | 主力净额 = 超大单净额 + 大单净额                            |
| net_pct_main    | 主力净占比(%)   | 主力净占比 = 主力净额 / 成交额                              |
| net_amount_xl   | 超大单净额(万)  | 超大单：大于等于50万股或者100万元的成交单                   |
| net_pct_xl      | 超大单净占比(%) | 超大单净占比 = 超大单净额 / 成交额                          |
| net_amount_l    | 大单净额(万)    | 大单：大于等于10万股或者20万元且小于50万股和100万元的成交单 |
| net_pct_l       | 大单净占比(%)   | 大单净占比 = 大单净额 / 成交额                              |
| net_amount_m    | 中单净额(万)    | 中单：大于等于2万股或者4万元且小于10万股和20万元的成交单    |
| net_pct_m       | 中单净占比(%)   | 中单净占比 = 中单净额 / 成交额                              |
| net_amount_s    | 小单净额(万)    | 小单：小于2万股和4万元的成交单                              |
| net_pct_s       | 小单净占比(%)   | 小单净占比 = 小单净额 / 成交额                              |

**返回**

返回一个 [pandas.DataFrame] 对象，默认的列索引为取得的全部字段. 如果给定了 fields 参数, 则列索引与给定的 fields 对应.

**示例**

```python
# 获取一只股票在一个时间段内的资金流量数据
get_money_flow('000001.XSHE', '2016-02-01', '2016-02-04')
get_money_flow('000001.XSHE', '2015-10-01', '2015-12-30', fields="change_pct")
get_money_flow(['000001.XSHE'], '2010-01-01', '2010-01-30', ["date", "sec_code", "change_pct", "net_amount_main", "net_pct_l", "net_amount_m"])

# 获取多只股票在一个时间段内的资金流向数据
get_money_flow(['000001.XSHE', '000040.XSHE', '000099.XSHE'], '2010-01-01', '2010-01-30')
# 获取多只股票在某一天的资金流向数据
get_money_flow(['000001.XSHE', '000040.XSHE', '000099.XSHE'], '2016-04-01', '2016-04-01')
```


===
数据导出要求：
我现在需要你帮我在聚宽中导出财务信息报表，导到 QLab 中进行因子挖掘。导出的 CSV 需满足以下要求：时间范围为2010年1月1日至2026年5月31日，股票池为中证500，具体股票列表已写在该文件中(`csi500_distinct.txt`)。文件里面存的代码格式是`SH600008`,你要自行转换为聚宽的那种股票代码格式。最后输出文件到`csi500_money_flow_20100101_202605.csv`。其中symbol 为股票代码。
文件格式
- 单个 CSV 文件，包含所有股票
- UTF-8 编码，逗号分隔

csv 有这么几个列
date,symbol,特征列1,特征列2,特征列3....


In [1]:
import os
import gc
import time
import pandas as pd
from jqdata import *
from IPython.display import display
from ipywidgets import IntProgress, HTML, VBox, Layout

# ============================================================
#  中证500资金流向数据导出 - 聚宽研究环境
# ============================================================

# ---- 配置 ----
INPUT_FILE  = 'csi500_distinct.txt'
OUTPUT_FILE = 'csi500_money_flow_20100101_202605.csv'
START_DATE  = '2010-01-01'
END_DATE    = '2026-05-31'

# 单次请求的数据量约为 BATCH_SIZE 只股票 * 1 年交易日。
# 如果聚宽环境内存紧张，可以把 BATCH_SIZE 调小到 5；速度优先可调到 20。
BATCH_SIZE  = 10
VERIFY_CHUNKSIZE = 200_000

FIELDS = [
    'date', 'sec_code', 'change_pct',
    'net_amount_main', 'net_pct_main',
    'net_amount_xl', 'net_pct_xl',
    'net_amount_l', 'net_pct_l',
    'net_amount_m', 'net_pct_m',
    'net_amount_s', 'net_pct_s',
]
FEATURE_COLUMNS = [c for c in FIELDS if c not in ('date', 'sec_code')]
OUTPUT_COLUMNS = ['date', 'symbol'] + FEATURE_COLUMNS

# ---- 进度条组件 ----
class ExportProgress:
    """Notebook 进度条组件：减少刷屏，同时展示速度、ETA、行数和错误样例。"""

    def __init__(self, total_steps, total_stocks):
        self.total_steps = total_steps
        self.total_stocks = total_stocks
        self.completed = 0
        self.rows = 0
        self.errors = 0
        self.first_errors = []
        self.start_time = time.time()
        self._last_render = 0

        bar_layout = Layout(width='680px')
        self.title = HTML(value='<h4 style="margin:2px 0">资金流向数据导出</h4>')
        self.total_label = HTML(value='总进度: 准备中...')
        self.total_bar = IntProgress(
            value=0,
            min=0,
            max=total_steps,
            layout=bar_layout,
            style={'bar_color': '#1976D2'},
        )
        self.current_label = HTML(value='当前批次: -')
        self.stats_label = HTML(value='')
        self.error_label = HTML(value='')
        self.ui = VBox(
            [self.title, self.total_label, self.total_bar, self.current_label, self.stats_label, self.error_label],
            layout=Layout(padding='10px', border='1px solid #ddd', border_radius='4px'),
        )
        display(self.ui)

    def update(self, current_desc='', new_rows=0, error=False, error_msg=''):
        self.completed += 1
        self.rows += new_rows
        if error:
            self.errors += 1
            if len(self.first_errors) < 8:
                self.first_errors.append(error_msg)

        now = time.time()
        if now - self._last_render >= 0.3 or self.completed >= self.total_steps:
            self._last_render = now
            self.render(current_desc)

    def render(self, current_desc=''):
        elapsed = time.time() - self.start_time
        speed = self.completed / elapsed if elapsed > 0 else 0
        eta = (self.total_steps - self.completed) / speed if speed > 0 else 0
        pct = self.completed / self.total_steps if self.total_steps else 1

        self.total_bar.value = self.completed
        self.total_label.value = (
            f'<b>总进度</b>: {self.completed:,}/{self.total_steps:,} ({pct:.1%}) | '
            f'股票 {self.total_stocks} 只 | {START_DATE} ~ {END_DATE}'
        )
        self.current_label.value = f'<b>当前批次</b>: {current_desc or "-"}'
        self.stats_label.value = (
            f'已写入 <b>{self.rows:,}</b> 行 | 错误 <b>{self.errors}</b> | '
            f'速度 {speed:.2f} 批/秒 | 已用 {self._fmt(elapsed)} | 剩余 {self._fmt(eta)}'
        )
        if self.first_errors:
            items = '<br>'.join(self.first_errors[-4:])
            self.error_label.value = f'<span style="color:#B00020">错误示例:<br>{items}</span>'
        else:
            self.error_label.value = ''

    def finish(self):
        self.render('完成')
        if self.errors == 0:
            self.title.value = '<h4 style="margin:2px 0;color:green">资金流向数据导出完成</h4>'
        else:
            self.title.value = '<h4 style="margin:2px 0;color:#B00020">资金流向数据导出完成，但有错误</h4>'
        print(f'导出完成: {self.rows:,} 行 -> {OUTPUT_FILE}')

    @staticmethod
    def _fmt(seconds):
        if seconds < 60:
            return f'{seconds:.1f}s'
        minutes, sec = divmod(int(seconds), 60)
        if minutes < 60:
            return f'{minutes}m {sec}s'
        hours, minutes = divmod(minutes, 60)
        return f'{hours}h {minutes}m'

# ---- 股票代码转换 ----
def read_raw_codes(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        codes = []
        for line in f:
            line = line.strip()
            if not line:
                continue
            codes.append(line.split('\t')[-1].strip())
    return codes


def to_jq_code(raw_code):
    """SH600008 -> 600008.XSHG, SZ000001 -> 000001.XSHE"""
    raw_code = raw_code.strip().upper()
    prefix, number = raw_code[:2], raw_code[2:]
    if prefix == 'SH':
        return f'{number}.XSHG'
    if prefix == 'SZ':
        return f'{number}.XSHE'
    raise ValueError(f'无法识别股票代码格式: {raw_code}')


def to_qlib_symbol(jq_code):
    """600008.XSHG -> sh600008, 000001.XSHE -> sz000001"""
    number, suffix = jq_code.split('.')
    return f'sh{number}' if suffix == 'XSHG' else f'sz{number}'

raw_codes = read_raw_codes(INPUT_FILE)
jq_codes = [to_jq_code(code) for code in raw_codes]
print(f'已加载股票数量: {len(jq_codes)}')
print(f'示例: {raw_codes[:3]} -> {jq_codes[:3]}')

# ---- API 连通性测试 ----
print('正在测试 get_money_flow API...')
test_df = get_money_flow('000001.XSHE', start_date='2020-01-02', end_date='2020-01-03', fields=FIELDS)
print(f'API 正常，测试返回 {len(test_df)} 行')
del test_df
gc.collect()

# ---- 初始化输出文件：只写表头，不保留任何历史内容 ----
pd.DataFrame(columns=OUTPUT_COLUMNS).to_csv(OUTPUT_FILE, index=False, encoding='utf-8')

# ---- 按年份拆分日期区间，控制单次返回数据量 ----
year_ranges = []
for year in range(2010, 2027):
    start = f'{year}-01-01'
    end = f'{year}-12-31' if year < 2026 else END_DATE
    year_ranges.append((start, end))

total_batches = (len(jq_codes) + BATCH_SIZE - 1) // BATCH_SIZE
total_steps = total_batches * len(year_ranges)
progress = ExportProgress(total_steps=total_steps, total_stocks=len(jq_codes))

# ---- 逐批拉取并立即追加写入：内存中只保留一个小批次 ----
for batch_idx, start_idx in enumerate(range(0, len(jq_codes), BATCH_SIZE), start=1):
    batch_codes = jq_codes[start_idx:start_idx + BATCH_SIZE]
    symbol_map = {code: to_qlib_symbol(code) for code in batch_codes}

    for start, end in year_ranges:
        desc = f'股票批次 {batch_idx}/{total_batches} | {batch_codes[0]}~{batch_codes[-1]} | {start}~{end}'
        try:
            df = get_money_flow(
                batch_codes,
                start_date=start,
                end_date=end,
                fields=FIELDS,
            )

            if df is None or df.empty:
                progress.update(current_desc=desc)
                continue

            # 只保留导出需要的列，立刻写入磁盘。
            df['date'] = pd.to_datetime(df['date']).dt.strftime('%Y-%m-%d')
            df['symbol'] = df['sec_code'].map(symbol_map)
            df = df.dropna(subset=['symbol'])
            df = df[OUTPUT_COLUMNS].sort_values(['symbol', 'date'])

            df.to_csv(OUTPUT_FILE, mode='a', header=False, index=False, encoding='utf-8')
            progress.update(current_desc=desc, new_rows=len(df))

            del df
        except Exception as e:
            msg = f'{desc} | {type(e).__name__}: {e}'
            progress.update(current_desc=desc, error=True, error_msg=msg)

        gc.collect()

progress.finish()

# ---- 输出文件快速验证：分块读取 date/symbol 两列，避免整表进内存 ----
print(f'\n验证文件: {OUTPUT_FILE}')
file_size_mb = os.path.getsize(OUTPUT_FILE) / (1024 * 1024)
row_count = 0
symbols = set()
min_date = None
max_date = None

for chunk in pd.read_csv(
    OUTPUT_FILE,
    usecols=['date', 'symbol'],
    chunksize=VERIFY_CHUNKSIZE,
    encoding='utf-8',
):
    row_count += len(chunk)
    symbols.update(chunk['symbol'].dropna().unique())
    cmin = chunk['date'].min()
    cmax = chunk['date'].max()
    min_date = cmin if min_date is None else min(min_date, cmin)
    max_date = cmax if max_date is None else max(max_date, cmax)
    del chunk

gc.collect()

print(f'文件大小: {file_size_mb:.1f} MB')
print(f'总行数: {row_count:,}')
print(f'股票数: {len(symbols)}')
print(f'日期范围: {min_date} ~ {max_date}')

# 样例只读取前 5 行，不读取全量 CSV。
display(pd.read_csv(OUTPUT_FILE, nrows=5, encoding='utf-8'))



已加载股票数量: 1712
示例: ['SH600004', 'SH600006', 'SH600007'] -> ['600004.XSHG', '600006.XSHG', '600007.XSHG']
正在测试 get_money_flow API...
API 正常，测试返回 2 行


导出完成: 5,612,489 行 -> csi500_money_flow_20100101_202605.csv

验证文件: csi500_money_flow_20100101_202605.csv
文件大小: 527.9 MB
总行数: 5,612,489
股票数: 1708
日期范围: 2010-01-04 ~ 2026-05-29


,date,symbol,change_pct,net_amount_main,net_pct_main,net_amount_xl,net_pct_xl,net_amount_l,net_pct_l,net_amount_m,net_pct_m,net_amount_s,net_pct_s
0,2010-01-04,sh600004,0.00,-457.47,-2.9,378.59,2.4,-836.06,-5.3,-141.97,-0.9,599.44,3.8
1,2010-01-05,sh600004,0.39,-236.63,-2.1,653.56,5.8,-890.20,-7.9,-225.37,-2.0,462.00,4.1
2,2010-01-06,sh600004,-1.47,-123.77,-1.3,66.64,0.7,-190.41,-2.0,-209.45,-2.2,333.22,3.5
3,2010-01-07,sh600004,-2.59,-1063.34,-12.0,-345.59,-3.9,-717.76,-8.1,53.17,0.6,1010.18,11.4
4,2010-01-08,sh600004,4.09,1784.64,11.9,1034.79,6.9,749.85,5.0,-404.92,-2.7,-1379.72,-9.2
